# noise_reachability step 14: is round 2's loss on 0/8 real, and is it dilution? (Kaggle, 2 x T4)

Step 14 of `noise_reachability/README.md`. Task 0 only. Round 1's adapters N0_s (step 10), round 2's R0_s (step 12) and
L0_s (R0_s's training set, trained for 2,671 steps instead of 2,000) are evaluated with sampled noise on fresh episodes
of 0/8 (all three) and of step 12's searched task-0 states (R and L).

**Needs two inputs:** Add Input > Your Work > the step 10 notebook, its **last** version (adapters N, round 1's
recordings), and the step 12 notebook (`expert-iteration-step-12`), its **last** version (adapters R, the search's
recordings). **Run with Save & Run All (Accelerator: GPU T4 x2, Internet on).** Nothing needs editing. The harness runs
first; a failure stops the run.

**This run needs about two sessions.** When step 14 says work remains, add this version's output as input as well
(**Add Input > Your Work**, only the latest step 14 version; keep step 10's and step 12's), and Save & Run All again.
Step 6 must then print `RESUMING`.

Download from the Output tab: `step14_results.zip` (results without recordings or adapter weights, which stay in the
version's output).

## 1. Settings

In [ ]:
import csv, glob, hashlib, json, os, re, shutil, signal, subprocess, sys, time
from pathlib import Path

T_SESSION = time.time()
REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "0f152b8e1c9d196cab39291991a7451181ff6d43"
assert REPO_REF != "SET_AFTER_COMMIT", "this notebook has no reviewed commit yet"
REPO = "/tmp/efficient_vla"        # code and venv live outside /kaggle/working, so they are not saved as output
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"

# The protocol of README.md step 14; nothing here is meant to be edited. The analysis refuses anything else.
CHECKPOINT_REPO, CHECKPOINT_REVISION = "lerobot/smolvla_libero", "a24f1f380461b665158d52acd140152c79e8b36d"
TASK = 0
N_ACTION_STEPS = 10
CHECK_STATE = 8                    # round 1's best trained state, where round 2 lost ground
SEARCHED = [2, 5, 6, 11]             # step 12's searched task-0 states
TRAIN_SEEDS = [0, 1, 2]
TRAIN_STEPS, BATCH_SIZE = 2671, 16       # 2,000 x 7,309 / 5,473 frames: round 1's samples per frame on R's set
HARNESS = {'N0': '16-19', 'R0': '48-51'}   # policy: seeds of its committed 0/8 episodes
CHECK_PARTS = ['80-95', '96-111', '112-127', '128-143', '144-159', '160-175']
SEARCHED_PARTS = ['80-83', '84-87', '88-91', '92-95']
CHECK_POLICIES = ['N0', 'N1', 'N2', 'R0', 'R1', 'R2', 'L0', 'L1', 'L2']
SEARCHED_POLICIES = ['R0', 'R1', 'R2', 'L0', 'L1', 'L2']
OUT_NAME = "step14"
OUT_ROOT = f"/kaggle/working/{OUT_NAME}"
ROUND2_DATA = "/tmp/step14"        # R's recordings (step 10's and step 12's), linked per session (not saved as output)
STOP_AT_H = 11.0                   # no new work after this many hours of the session; running jobs are stopped
TRAIN_NEEDS_H = 5.0                # before the timing check: no training run starts with less time left
GB_PER_JOB = 6                     # base policy + one episode in RAM per simulator process

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"
ADAPTERS = [f"L{TASK}_{s}" for s in TRAIN_SEEDS]
print(f"step 14 at {REPO_REF}: task {TASK}, check state {CHECK_STATE}, searched {SEARCHED}, adapters {ADAPTERS} "
      f"({TRAIN_STEPS} steps), at most 1256 episodes (8 harness, 864 on "
      f"0/8, 384 on the searched states), stop after {STOP_AT_H} h, output {OUT_ROOT}")


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r


def hours_left():
    return STOP_AT_H - (time.time() - T_SESSION) / 3600

## 2. Hardware

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader")
gpus = [line.split(", ") for line in smi.stdout.strip().splitlines()]
assert gpus, "No GPU. Set Accelerator to GPU T4 x2."
for name, mem, cap, driver in gpus:
    assert float(cap) >= 7.5, f"{name} (compute capability {cap}) is not supported by the pinned PyTorch build; use GPU T4 x2"
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
mem_gb = int(open("/proc/meminfo").read().split()[1]) / 1e6
avail_gb = int(re.search(r"MemAvailable:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
JOBS = max(1, min(2 * len(gpus), int((avail_gb - 4) // GB_PER_JOB), os.cpu_count()))
GPU_IDS = ",".join(str(i) for i in range(len(gpus)))
print(f"{len(gpus)} GPU(s), driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; "
      f"RAM {mem_gb:.0f} GB ({avail_gb:.0f} GB available) -> {JOBS} evaluations at a time on GPUs {GPU_IDS}")
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"
assert JOBS >= 4, f"RAM for {JOBS} SmolVLA processes; the notebook runs 4 at once"

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

The package versions of the local `lerobot_env` (as the earlier Kaggle notebooks). Takes several minutes.

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks, tests and the base checkpoint

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", OMP_NUM_THREADS="1", PYTHONPATH=REPO,
               PYTHONFAULTHANDLER="1")
sh(f"""{PY} -c "
import torch, av
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], '| av', av.__version__)
" """, env=RUN_ENV)
egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
assert "EGL context ok" in egl.stdout, "EGL rendering failed"
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)
r = sh(f"""{PY} -c "from huggingface_hub import snapshot_download; print(snapshot_download('{CHECKPOINT_REPO}', revision='{CHECKPOINT_REVISION}'))" """, env=RUN_ENV)
POLICY_PATH = r.stdout.strip().splitlines()[-1]
print("base checkpoint", POLICY_PATH)

## 6. Resume from an earlier version's output

In [ ]:
os.makedirs(OUT_ROOT, exist_ok=True)
previous = [p for p in sorted(glob.glob(f"/kaggle/input/**/{OUT_NAME}", recursive=True))
            if os.path.isdir(p) and os.path.exists(f"{p}/repo_commit.txt")]
assert len(previous) <= 1, f"attach only the latest version's output, found {previous}"
for prev in previous:
    print("RESUMING from", prev)
    shutil.copytree(prev, OUT_ROOT, dirs_exist_ok=True)
if not previous:
    print("FRESH START: no earlier output attached. If this is the second session, stop and add it as input.", flush=True)
head = sh(f"git -C {REPO} log -1 --format='%H'").stdout.strip()
first = Path(OUT_ROOT, "repo_commit.txt")
started_with = first.read_text().split()[0] if first.exists() and first.read_text().strip() else None
if started_with and started_with != head:
    raise RuntimeError(f"the run started at commit {started_with}, this version cloned {head}: set REPO_REF in "
                       f"step 1 to {started_with}")
with open(first, "a") as f:
    f.write(head + "\n")
for d in ("logs", "eval"):
    os.makedirs(f"{OUT_ROOT}/{d}", exist_ok=True)

## 7. Inputs: step 10's output (N, round 1's recordings) and step 12's (R, the search's recordings)

In [ ]:
def find_input(name, marker):
    found = [p for p in sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)) if os.path.exists(f"{p}/{marker}")]
    assert len(found) == 1, f"attach {name}'s last version as input (found {found})"
    return found[0]


STEP10_IN = find_input("step10", f"runs/N{TASK}_0/adapter/run_config.json")
STEP12_IN = find_input("step12", f"runs/R{TASK}_0/adapter/run_config.json")
for name, path in (("step 10", STEP10_IN), ("step 12", STEP12_IN)):
    print(name, "output:", path, "| its commits:", open(f"{path}/repo_commit.txt").read().split())
for s in TRAIN_SEEDS:
    assert os.path.exists(f"{STEP10_IN}/runs/N{TASK}_{s}/adapter/run_config.json"), f"step 10 has no adapter N{TASK}_{s}"
    assert os.path.exists(f"{STEP12_IN}/runs/R{TASK}_{s}/adapter/run_config.json"), f"step 12 has no adapter R{TASK}_{s}"


def n_adapter(s):
    return f"{STEP10_IN}/runs/N{TASK}_{s}/adapter"


def r_adapter(s):
    return f"{STEP12_IN}/runs/R{TASK}_{s}/adapter"

## 8. Helpers: parallel jobs with resume and a deadline

In [ ]:
def run_jobs(jobs, max_parallel, poll_s=30, log_every_s=600):
    """Run {name: (command, gpu)} with at most max_parallel at once; exit code 3 means 'call again' (more
    episodes in a fresh process). No job starts after the deadline and running ones are stopped at it.
    Returns {name: exit code}; a job stopped at the deadline gets None."""
    pending, running, codes, last_log = list(jobs), {}, {}, time.time()
    while pending or running:
        while pending and len(running) < max_parallel and hours_left() > 0:
            name = pending.pop(0)
            cmd, gpu = jobs[name]
            log = open(f"{OUT_ROOT}/logs/{name}.log", "a")
            env = dict(RUN_ENV, CUDA_VISIBLE_DEVICES=str(gpu))
            running[name] = (subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=env, stdout=log,
                                              stderr=subprocess.STDOUT, start_new_session=True), log)
            time.sleep(5)  # staggered EGL start-up (FAIL-006)
        if hours_left() <= 0 and running:
            print(time.strftime("%H:%M"), "deadline: stopping", list(running), flush=True)
            for name, (proc, log) in running.items():
                if proc.poll() is None:
                    os.killpg(proc.pid, signal.SIGTERM)
            time.sleep(20)
            for name, (proc, log) in running.items():
                log.close()
                codes[name] = None
            for name in pending:
                codes[name] = None
            return codes
        if hours_left() <= 0 and not running:
            for name in pending:
                codes[name] = None
            return codes
        for name, (proc, log) in list(running.items()):
            if proc.poll() is None:
                continue
            log.close()
            del running[name]
            if proc.returncode == 3:
                pending.insert(0, name)
                continue
            codes[name] = proc.returncode
            print(time.strftime("%H:%M"), name, "exit", proc.returncode, flush=True)
        if time.time() - last_log > log_every_s:
            last_log = time.time()
            print(time.strftime("%H:%M"), f"{hours_left():.1f} h left; running {list(running)}; "
                  f"{len(pending)} waiting", flush=True)
        time.sleep(poll_s)
    return codes


FIELDS = ["task_id", "seed", "init_state", "success", "steps"]


def read_csv(path):
    return list(csv.DictReader(open(path))) if os.path.exists(path) else []


def write_csv(rows, out):
    os.makedirs(os.path.dirname(out), exist_ok=True)
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows({k: r[k] for k in FIELDS} for r in rows)
    return rows


def n_range(spec):
    a, b = (spec.split("-") + [spec])[:2]
    return int(b) - int(a) + 1


def failed(codes):
    return {k: c for k, c in codes.items() if c not in (0, None)}


FAILURE = None


def fail(message):
    """Keep going to the zip in step 15, which raises this."""
    global FAILURE
    FAILURE = message
    print("FAILED:", message, flush=True)






ANALYZE = f"cd {REPO} && {PY} -S noise_reachability/analyze_step14.py --root {OUT_ROOT}"
MAIN = f"{REPO}/self_filtered_bc/self_filtered_bc.py"


def sfbc():
    return f"cd {REPO} && {PY} -u {MAIN} --task_id {TASK} --n_action_steps {N_ACTION_STEPS} --policy_path {POLICY_PATH}"


def run_stage(name, jobs, total, max_parallel=4):
    """True when the stage has no work left: a job that failed stops the run, the deadline leaves work for a next version."""
    print(f"{name}: {len(jobs)} of {total} jobs to run; {hours_left():.1f} h left", flush=True)
    if not jobs:
        return True
    if hours_left() <= 0:
        return False
    codes = run_jobs(jobs, max_parallel=max_parallel)
    if failed(codes):
        fail(f"{name} failed: {failed(codes)}")
        return False
    return all(c == 0 for c in codes.values()) and len(codes) == len(jobs)


def run_gate(which):
    if sh(f"{ANALYZE} --gate {which}", check=False, env=RUN_ENV).returncode != 0:
        fail(f"the {which} gate failed: investigate before spending more")
        return False
    return True


def adapter_done(name):
    return Path(OUT_ROOT, "runs", name, "adapter", "run_config.json").exists()


def recordings(folder):
    return [f for f in glob.glob(f"{folder}/episodes/*.npz") if not f.endswith(".tmp.npz")]

## 9. Harness: N0_0 and R0_0 on four of their own committed episodes of 0/8 each; stops the run if it fails

At least 7 of the 8 episodes must reproduce the committed rows exactly (success and number of steps).

In [ ]:
HARNESS_DONE = False
jobs = {}
for k, (policy, seeds) in enumerate(HARNESS.items()):
    out = f"{OUT_ROOT}/harness/{policy}"
    if len(read_csv(f"{out}/per_episode_lora.csv")) < n_range(seeds):
        lora = n_adapter(policy[1:]) if policy.startswith("N") else r_adapter(policy[1:])
        jobs[f"harness_{policy}"] = (f"{sfbc()} --stage eval --eval_policies lora --lora_dir {lora} --eval_seeds {seeds} "
                                     f"--eval_init_states {CHECK_STATE} --eval_output_dir {out} "
                                     f"--max_episodes_per_process 10", k % 2)
if run_stage("harness", jobs, len(HARNESS)):
    HARNESS_DONE = run_gate("harness")

## 10. Evaluation helpers: 0/8 (N, R, L) and the searched states (R, L), sampled noise, fresh seeds

In [ ]:
SETS = {"eval_check": (str(CHECK_STATE), CHECK_PARTS, CHECK_POLICIES),
        "eval_searched": (",".join(map(str, SEARCHED)), SEARCHED_PARTS, SEARCHED_POLICIES)}


def eval_out(which, policy, j):
    return f"{OUT_ROOT}/{which}/{policy}/part{j}"


def lora_of(policy):
    s = policy[1:]
    return {"N": n_adapter(s), "R": r_adapter(s), "L": f"{OUT_ROOT}/runs/L{TASK}_{s}/adapter"}[policy[0]]


def eval_cmd(which, policy, j):
    states, parts, _ = SETS[which]
    return (f"{sfbc()} --stage eval --eval_policies lora --lora_dir {lora_of(policy)} --eval_seeds {parts[j]} "
            f"--eval_init_states {states} --eval_output_dir {eval_out(which, policy, j)} --max_episodes_per_process 10")


def part_size(which, j):
    states, parts, _ = SETS[which]
    return n_range(parts[j]) * len(states.split(","))


def eval_rows(which, policy):
    return [r for j in range(len(SETS[which][1])) for r in read_csv(f"{eval_out(which, policy, j)}/per_episode_lora.csv")]


def evaluate(letters):
    """Every unfinished part of the policies whose first letter is in ``letters`` and whose adapter is ready."""
    jobs, k = {}, 0
    for which, (_, parts, policies) in SETS.items():
        for policy in policies:
            if policy[0] not in letters or (policy[0] == "L" and not adapter_done(f"L{TASK}_{policy[1:]}")):
                continue
            for j in range(len(parts)):
                if len(read_csv(f"{eval_out(which, policy, j)}/per_episode_lora.csv")) < part_size(which, j):
                    jobs[f"{which}_{policy}_part{j}"] = (eval_cmd(which, policy, j), k % 2)
                    k += 1
    run_stage(f"evaluation of {letters}", jobs, len(jobs))

## 11. Evaluation of N0-N2 and R0-R2 (their adapters exist already)

In [ ]:
if HARNESS_DONE and not FAILURE:
    evaluate("NR")

## 12. L's training data: exactly R0_s's recordings (step 10's task-0 recordings and step 12's task-0 search successes)

Linked (not copied) under `/tmp/step14/task_0/round2_data/episodes/` in every session; training keeps the successes. The
analysis checks that each L0_s trained on exactly R0_s's episodes.

In [ ]:
def round2_data():
    return f"{ROUND2_DATA}/task_{TASK}/round2_data"


if HARNESS_DONE and not FAILURE:
    dest = f"{round2_data()}/episodes"
    shutil.rmtree(dest, ignore_errors=True)
    os.makedirs(dest)
    old, new = recordings(f"{STEP10_IN}/task_{TASK}/replay_data"), recordings(f"{STEP12_IN}/task_{TASK}/search_data")
    for f in old + new:
        target = f"{dest}/{os.path.basename(f)}"
        assert not os.path.exists(target), f"two recordings named {os.path.basename(f)}"
        os.symlink(f, target)
    print(f"task {TASK}: {len(old)} round-1 and {len(new)} round-2 recordings (successes and failures)")
    # before about 12 GPU-hours of training: the selection must give exactly R0_s's committed training episodes
    if sh(f"cd {REPO} && {PY} noise_reachability/check_step14_data.py {round2_data()}", check=False,
          env=RUN_ENV).returncode != 0:
        fail("L's training data is not R's: check the step 10 and step 12 inputs (last versions?)")

## 13. Training: L0_s from the base for 2,671 steps, training seeds 0-2

Two adapters at a time, one per T4. A run starts only if the session can finish it (1.25 x the measured time plus 10
minutes); an unfinished run restarts from scratch in the next session.

In [ ]:
TIMING = Path(OUT_ROOT, "train_timing.json")
if HARNESS_DONE and not FAILURE and not TIMING.exists() and hours_left() > 0.5:
    r = sh(f"{sfbc()} --stage train --seed 0 --selection success --data_dir {round2_data()} --train_steps 20 "
           f"--batch_size {BATCH_SIZE} --log_every 5 --val_every 1000 --output_dir /tmp/step14_train_timing",
           check=False, env=dict(RUN_ENV, CUDA_VISIBLE_DEVICES="0"))
    found = re.findall(r"sec_per_step=([0-9.]+)", r.stdout)
    if r.returncode != 0 or not found:
        fail("the training timing run failed")
    else:
        TIMING.write_text(json.dumps({"sec_per_step": float(found[-1]), "date": time.strftime("%Y-%m-%d %H:%M")}) + "\n")
if TIMING.exists():
    TRAIN_NEEDS_H = 1.25 * TRAIN_STEPS * json.loads(TIMING.read_text())["sec_per_step"] / 3600 + 10 / 60
print(f"a training run starts only with more than {TRAIN_NEEDS_H:.2f} h left")


def train_cmd(name):
    s = name.split("_")[1]
    return (f"{sfbc()} --stage train --selection success --train_steps {TRAIN_STEPS} --batch_size {BATCH_SIZE} "
            f"--seed {s} --data_dir {round2_data()} --output_dir {OUT_ROOT}/runs/{name}")


ready = [n for n in ADAPTERS if not adapter_done(n)] if HARNESS_DONE and not FAILURE else []
print("adapters to train now:", ready or "none", flush=True)
while ready and hours_left() > TRAIN_NEEDS_H and not FAILURE:
    batch, ready = ready[:2], ready[2:]
    for n in batch:
        shutil.rmtree(f"{OUT_ROOT}/runs/{n}", ignore_errors=True)   # an unfinished run restarts from scratch
    codes = run_jobs({f"train_{n}": (train_cmd(n), k) for k, n in enumerate(batch)}, max_parallel=2)
    if failed(codes):
        fail(f"training failed: {failed(codes)}")
    for n in batch:
        lines = open(f"{OUT_ROOT}/logs/train_{n}.log").read().splitlines()
        print(n, [l for l in lines if "training episodes" in l or "training frames" in l or "peak GPU" in l])
DONE_ADAPTERS = [n for n in ADAPTERS if adapter_done(n)]
print("adapters done:", DONE_ADAPTERS)

## 14. Evaluation of every L whose adapter is ready; then what is left

In [ ]:
if HARNESS_DONE and not FAILURE:
    evaluate("L")
for which, (_, parts, policies) in SETS.items():
    print(which, {p: len(eval_rows(which, p)) for p in policies}, f"of {sum(part_size(which, j) for j in range(len(parts)))} each")
ALL_DONE = (len(DONE_ADAPTERS) == len(ADAPTERS) and all(
    len(eval_rows(w, p)) == sum(part_size(w, j) for j in range(len(SETS[w][1]))) for w in SETS for p in SETS[w][2]))
if ALL_DONE and not FAILURE:
    if sh(f"{ANALYZE} --out {OUT_ROOT}/step14_report.txt", check=False, env=RUN_ENV).returncode != 0:
        fail("the analysis failed: the episodes are in the zip")
if not ALL_DONE and not FAILURE:
    print("WORK REMAINS: add this version's output as input (keep step 10's and step 12's) and Save & Run All again")

## 15. Files for the repo

In [ ]:
sh(f"cd '{OUT_ROOT}' && zip -q -r /kaggle/working/step14_results.zip . -x '*.npz' '*.safetensors' '*.pt'")
sh(f"du -h /kaggle/working/step14_results.zip; du -sh {OUT_ROOT}")
print(f"session time used: {(time.time() - T_SESSION) / 3600:.1f} h")
if FAILURE:
    raise RuntimeError(FAILURE)